In [18]:
import pandas as pd
df = pd.read_csv("results.tsv", sep="\t", index_col=0) 
labels = pd.read_csv("gencode.v34.types.tsv", sep="\t", index_col=0)

In [19]:
df["baseMean"].mean(skipna = True)

282.3899195523489

In [20]:
#df.drop('Unnamed: 6', inplace=True, axis=1)
#df["FoldChange"] = 2 ** df["log2FoldChange"] 

#inner_merged_total = pd.merge(df, labels, on=["Type"])
#df_labels = pd.merge(df, labels, on=["Type"])
#df.merge(labels, how = "left",)
#pd.merge(df, labels, how="left")
#df.columns

In [21]:
df = df.sort_values(by = ["pvalue"], axis = 0, ascending = True)
df = df.sort_values(by = ["log2FoldChange"], key = abs, axis = 0, ascending = False)
df.insert(1, "FoldChange", 2 ** df["log2FoldChange"], allow_duplicates = False)

In [22]:
df.head(10)
#df.iloc[10000]
#lfcSE standart deviation of log2FoldChange

,baseMean,FoldChange,log2FoldChange,lfcSE,pvalue,padj
TUSC3,33.887663,824.354639,9.687121,2.813555,4.054186e-13,1.708383e-12
HMOX1,16880.360560,431.307037,8.752571,0.095777,0.000000e+00,0.000000e+00
COL4A1,109.287534,348.267340,8.444051,0.969371,4.537307e-17,2.245169e-16
HSPA6,186.202662,320.814597,8.325596,0.709425,1.071676e-30,8.211715e-30
NLRP4,42.211912,230.014964,7.845584,1.293231,1.351432e-11,5.312414e-11
RBPMS2,317.415626,228.487264,7.835970,0.461536,1.025356e-63,1.555055e-62
KRT89P,15.678446,0.004743,-7.719843,2.686689,1.067215e-08,3.622614e-08
MAP1A,3009.589442,202.805284,7.663951,0.146464,0.000000e+00,0.000000e+00
HTRA3,10.747456,202.391328,7.661004,2.650903,3.440312e-08,1.135809e-07
DRAIC,14.509998,0.005304,-7.558829,2.671096,2.275844e-08,7.587711e-08


In [23]:
df_select = df.loc[(df["padj"] < 0.05)]

df_select_up = df_select.loc[(df["log2FoldChange"] >= 1)] # (2926, 6)
df_select_up.to_csv("hypoxie_FCstat_up.tsv", sep="\t")

df_select_down = df_select.loc[(df["log2FoldChange"] <= -1)] # (3866, 6)
df_select_down.to_csv("hypoxie_FCstat_down.tsv", sep="\t")

# in search for target genes
possible_target = pd.concat([df_select_up, df_select_down], axis = 0)

possible_target = possible_target.loc[(df["baseMean"] > 282.38992)]
possible_target = possible_target.sort_values(by = ["log2FoldChange"], key = abs, axis = 0, ascending = False)
for col_name, data in possible_target.items():
    possible_target[col_name] = round(possible_target[col_name], 3)

    
possible_target.to_csv("hypoxie_possible_target.tsv", sep="\t")

In [24]:
# in search for ref genes
import statistics as st
st.median(df["baseMean"])

#print(df["baseMean"].median(skipna = True)) # медиана baseMean
print(df["baseMean"].mean(skipna = True)) # среднее занчение baseMean 282.38991955231324

possible_ref = df.loc[(df["log2FoldChange"] < 1) & (df["log2FoldChange"] > -1) & (df["baseMean"] > 282.38992)]
possible_ref = possible_ref.sort_values(by = ["log2FoldChange"], key = abs, axis = 0, ascending = True)
#print(possible_ref.shape) # (5553, 6); (39, 6) если min baseMean = 17647


for col_name, data in possible_ref.items():
    possible_ref[col_name] = round(possible_ref[col_name], 3)

possible_ref.to_csv("hypoxie_possible_refer.tsv", sep="\t")

#possible_ref.head(20)

282.38991955234883


In [25]:
possible_ref.loc['ACTB'] # ACTB is a common reference gene: mean is 17647 

baseMean          17647.742
FoldChange            0.750
log2FoldChange       -0.415
lfcSE                 0.056
pvalue                0.000
padj                  0.000
Name: ACTB, dtype: float64

In [26]:
for i in ["HPRT1"]:
    try:
        print(possible_ref.loc[i]) 
    except KeyError:
        print(f" {i} not in df")

baseMean          453.558
FoldChange          0.995
log2FoldChange     -0.008
lfcSE               0.102
pvalue              0.936
padj                0.956
Name: HPRT1, dtype: float64


In [27]:
for i in ["ACTB"]:
    try:
        print(possible_target.loc[i]) 
    except KeyError:
        print(f" {i} not in df")

 ACTB not in df
